# Imports

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os, sys, time
import matplotlib.pyplot as plt
from pathlib import Path

from molmod.units import *
from molmod.constants import *
from molmod.periodic import periodic as pp
from yaff import log as ylog, PREOS, System as YaffSystem, ForceField
ylog.set_level(ylog.silent)

from cmmdft.log import log
from cmmdft.system import EmptyHost, NanoporousHost, Guest, Host, Grid, SphericalLJGuest
from cmmdft.program import Program
from cmmdft.eos import VanderWaalsEOS as VdW, ModifiedBenedictWebbRubinEOS as MBWR, MFMT_MFA_EOS as MME
from cmmdft.plotter import Plotter, MultiPlotter
from cmmdft.calculator import Calculator
from cmmdft.DensityEOS import density_from_EOS
from cmmdft.tools import selection_sort, potential_from_mfa, find_local_maxima

#%matplotlib widget
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

%matplotlib widget

# System specifications

In [2]:
# Host
hostname = 'ZIF8'
hostchk = 'InputFiles/Host/ZIF8/struct.chk'
hostpar = 'InputFiles/Host/ZIF8/pars_vdw_uff.txt'

guestname = 'CH4_UA'
guestchk = 'InputFiles/Guest/CH4/struct_UA.chk'
guestpar = 'InputFiles/Guest/CH4/pars_vdw_UA_trappe.txt'
sigma, epsilon = 3.75*angstrom, 0.29411*kcalmol
mass = 16.04*amu

## GRID SPECIFICATIONS
#npoints=[60,50,50]
spacing = 0.5*angstrom

## OUTPUT DESTINATION
ff_suffix = 'UFF_TRAPPE'
funct_suffix = 'MFMT_MFA_WDA_c'
grid_suffix = '0_5'

# Program setup

In [3]:
temperature = 300*kelvin

#initialize program
program = Program(prefix='OutputFiles', hostname=hostname, guestname=guestname, ff_suffix=ff_suffix, funct_suffix=funct_suffix,
                  grid_suffix=grid_suffix, overwrite=False)

guest = SphericalLJGuest(guestname, mass, sigma, epsilon)
program.set_system(NanoporousHost(hostname, hostchk, hostpar), guest)
program.set_grid(spacing=spacing)

#construct the free energy functional
program.init_free_energy(temperature)
program.fener.add_external_potential()
program.fener.add_mean_field()
program.fener.add_hard_sphere(version='MFMT')
program.fener.add_correlation_wda_lj()

program.fener.set_temperature(temperature)

Created work directory OutputFiles/ZIF8/CH4_UA/UFF_TRAPPE/MFMT_MFA_WDA_c/0_5

********************************************************************************

                            Welcome to CmmDFT
    a Python package for computing density profiles using classical DFT

                               Written by
            Louis Vanduyfhuys(1) and Vic De Ridder(1)* and Steven Vandenbrande(1)

         (1) Center for Molecular Modeling, Ghent University Belgium.
                   * mailto: Vic.DeRidder@UGent.be

********************************************************************************


 USER               vjdridde
 MACHINE            Linux DESKTOP-C47ULF4 5.15.133.1-microsoft-standard-WSL2 #1 SMP Thu Oct 5 
 MACHINE            21:02:42 UTC 2023 x86_64
 TIME               2025-03-04 11:45:24.895267
 CMMDFT VERSION     0.1
 PYTHON VERSION     3.8.16 | packaged by conda-forge | (default, Feb  1 2023, 16:01:55) [GCC 11.3.0]
 NUMPY VERSION      1.24.4
 SCIPY VERSION      1.1

# Setting the simulation condition

In [ ]:
from yaff.pes.eos import PREOS
eos = PREOS.from_name('methane')

pressures = np.linspace(1e-4,65,30)*bar
chempots = [eos.calculate_mu(temperature,P) for P in pressures]

# Running the cDFT simulation

In [ ]:
fn = 1e-5

for mu in chempots:
    program.solve(chempot=mu, Ninit=fn, energy_tracking=True, nsteps = 400, alpha_mix=0.1, threshold=5e-7, method='hybrid', rewrite=False)
    # fn = '%s/rho_%4.5fkJmol_%3.0fK.npy' %(workdir,mu/kjmol,temperature/kelvin)
    fn = '%s/rho_%7.5fkJmol_%7.5fK.npy' %(program.workdir,mu/kjmol,temperature/kelvin)

# Saving the calculated loadings

In [ ]:
calc = Calculator(program)

#save loadings in .csv file
calc.save_loadings(300, chempots = chempots)
calc.save_loadings(300, chempots = chempots, pressure=True, eos=eos)